# FlyLM · MaleCNS 신경 동역학과 내부 기억

전체 `Traced` 뉴런과 그 사이의 공개 연결을 보존하는 byte 언어모델.
입력은 실제 sensory annotation, 출력은 실제 motor annotation을 따른다.
현재 문자 → 작은 감각 변환기 → MaleCNS 회로 → 작은 운동 출력기 → 다음 문자.
질문 재주입·문맥 pooling·외부 언어모델·새로운 뉴런 간 연결은 없다.

뉴런 상태는 막 상태 `v`, 비음수 시냅스 신호 `s`, 느린 적응 `a` 세 값이다.
회로의 시간적 문맥은 이 상태에만 저장한다. 생물학적 기억 재현이나 대화 성능은 아직 검증되지 않았다.
뉴런 유형별 시간 상수/역치/적응 강도를 학습하지만 초기값·범위는 공학적 가정이다.
시간 단위는 simulation tick이며 ms에 대응한다고 주장하지 않는다.
역전파/Adam을 사용한다. 도파민 가소성·개별 수용체·가지별 구획은 이번 구현에 포함하지 않는다.

[MaleCNS 공식 데이터](https://male-cns.janelia.org/download/) ·
[동역학과 과제 최적화를 결합한 선행 연구](https://www.nature.com/articles/s41586-024-07939-3)

**Colab GPU에서 위에서 아래로 실행. 기존 checkpoint와 호환되지 않으므로 새로 학습한다.**
기본 1,000 update는 실행 확인용이며 학습 완료 기준이 아니다. 대조군 실험은 포함하지 않는다.

## 1. 설정

전체 그래프 / batch 1 / TBPTT 64 / 문자당 4 simulation ticks / FP32.
큰 그래프는 T4 메모리 검사를 통과해야 실행된다. 자동 뉴런/연결 축소는 하지 않는다.
한 번에 64단계만 역전파하며, 그 이전 문맥은 상태 값으로 전달하되 기울기는 끊는다.
첫 답변 학습 구간은 최대 마지막 32 prompt bytes를 포함한다.

In [ ]:
import importlib.util
import subprocess
import sys

required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
            'pyarrow': 'pyarrow', 'tqdm': 'tqdm'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('CUDA PyTorch가 제공되는 Colab GPU 런타임을 선택하세요.')

import gc
import hashlib
import json
import math
import platform
import random
import shutil
import time
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import scipy
from scipy import sparse
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

CFG = dict(seed=42, n_neurons=None, embedding_dim=32, brain_steps=4,
           batch_size=1, tbptt=64, prompt_grad_tokens=32,
           train_updates=1000, epochs=1, lr=0.001,
           checkpoint_tokens=True, nt_confidence=0.8, use_nt_signs=True,
           edge_block=262144,
           tau_initial=(2.0, 2.0, 64.0),
           tau_min=(0.5, 0.5, 8.0), tau_max=(32.0, 16.0, 512.0),
           log_every=50, eval_every=200, save_every=200,
           eval_max_pairs=4, eval_response_tokens=128)
CACHE = Path('flylm_malecns_cache')
CACHE.mkdir(exist_ok=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if not torch.cuda.is_available():
    raise RuntimeError('이 노트북은 원격 CUDA GPU 런타임용입니다. T4를 선택하고 다시 실행하세요.')
torch.cuda.manual_seed_all(CFG['seed'])
device = torch.device('cuda')
VERSIONS = dict(python=platform.python_version(), torch=torch.__version__,
                numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__, pyarrow=pa.__version__)
print(VERSIONS)
print(torch.cuda.get_device_name(0), 'GPU GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
assert CFG['tbptt'] == 64
assert CFG['batch_size'] == 1, '입력–답변 경계를 보존하는 현재 로더는 batch 1 기준입니다.'

assert CFG['epochs'] >= 1 and CFG['tbptt'] >= 1
assert CFG['train_updates'] is None or CFG['train_updates'] >= 1
assert min(CFG[k] for k in ['log_every', 'eval_every', 'save_every', 'eval_max_pairs', 'eval_response_tokens']) >= 1
assert 1 <= CFG['prompt_grad_tokens'] < CFG['tbptt']

## 1A. Drive

/content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip을 사용한다.
파일명이 macOS NFD 한글로 저장된 경우에만 같은 디렉터리에서 Unicode 정규화한 동일 파일명을 찾아 실제 경로를 출력한다.
원본 ZIP은 수정하지 않으며 VM 로컬 복사본을 읽는다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import unicodedata
DATA_CFG = dict(zip_path='/content/drive/MyDrive/text_dataset/dataset/일상대화_데이터.zip',
                text_field='form', validation_fraction=0.05)
requested_zip = Path(DATA_CFG['zip_path'])
ZIP_PATH = requested_zip if requested_zip.is_file() else None
if ZIP_PATH is None and requested_zip.parent.is_dir():
    matches = [p for p in requested_zip.parent.iterdir() if p.is_file()
               and unicodedata.normalize('NFC', p.name) == requested_zip.name]
    if len(matches) == 1:
        ZIP_PATH = matches[0]
if ZIP_PATH is None:
    raise FileNotFoundError(f'ZIP을 확인하세요: {requested_zip}')
print('학습 데이터 ZIP:', ZIP_PATH)
DATA_CACHE = Path('/content/flylm_dialogue_cache')
DATA_CACHE.mkdir(exist_ok=True)
local_zip = DATA_CACHE / 'dialogues.zip'
stamp_path = DATA_CACHE / 'zip_source.json'
zip_stat = ZIP_PATH.stat()
stamp = dict(path=str(ZIP_PATH), size=zip_stat.st_size, mtime_ns=zip_stat.st_mtime_ns)
old_stamp = json.loads(stamp_path.read_text()) if stamp_path.exists() else None
if old_stamp != stamp or not local_zip.exists() or local_zip.stat().st_size != zip_stat.st_size:
    if shutil.disk_usage(DATA_CACHE).free < zip_stat.st_size + 512 * 1024**2:
        raise RuntimeError('ZIP 로컬 복사본을 위한 디스크 공간이 부족합니다.')
    partial = local_zip.with_suffix('.partial')
    shutil.copyfile(ZIP_PATH, partial)
    if partial.stat().st_size != zip_stat.st_size:
        raise RuntimeError('ZIP 복사 크기가 일치하지 않습니다.')
    partial.replace(local_zip)
    stamp_path.write_text(json.dumps(stamp), encoding='utf-8')
digest = hashlib.sha256()
with local_zip.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024**2), b''):
        digest.update(block)
zip_sha256 = digest.hexdigest()
print('ZIP SHA-256:', zip_sha256)


## 1B. 두 화자의 입력–답변 쌍과 문서 단위 split

JSON 배열 순서를 유지한다. form의 연속 공백만 정리하고 original_form/인구통계/시간 metadata는 학습 텍스트에 넣지 않는다.
text_field를 original_form으로 바꾸면 그 필드로 다시 전처리한다. 지정 필드의 누락/잘못된 타입은 위치를 표시하고 중단한다.
speaker_id로 화자를 구분하며 동일 화자의 연속 발화를 하나로 합친다. 마지막 화자1에 답변이 없으면 해당 turn은 쓰지 않는다.

각 학습 쌍은 다음 구조다.

    BOS + "화자1: 입력 내용\n화자2: " + 답변 내용 + EOS
    └──────────── loss 제외 ───────────┘  └── loss 적용 ──┘

쌍마다 prompt_tokens 경계를 기록한다. uint16 토큰 캐시에 입력과 답변을 저장하되, prompt는 label -100으로 mask한다.
중복 대화/쌍을 제거한 후 대화 단위 95:5 분리를 seed와 hash로 고정한다.
같은 화자가 다른 문서에도 있을 수 있으므로 화자 독립 평가는 아니다.
최소 2개 유효 대화가 필요하며, 조건에 맞는 쌍이 없으면 가짜 답변을 만들지 않고 중단한다.


In [ ]:
def dialogue_turns(document, location):
    utterances = document.get('utterance')
    if not isinstance(utterances, list):
        raise ValueError(f'{location}: utterance must be a list')
    speakers, turns = {}, []
    for index, utterance in enumerate(utterances):
        if not isinstance(utterance, dict):
            raise ValueError(f'{location}: utterance[{index}] must be an object')
        text = utterance.get(DATA_CFG['text_field'])
        if not isinstance(text, str):
            raise ValueError(f'{location}: utterance[{index}].{DATA_CFG["text_field"]} must be a string')
        text = ' '.join(text.split())
        if not text:
            continue
        speaker = utterance.get('speaker_id')
        if not isinstance(speaker, str) or not speaker.strip():
            return [], 'missing_speaker'
        if speaker not in speakers:
            speakers[speaker] = len(speakers) + 1
        role = speakers[speaker]
        if turns and turns[-1][0] == role:
            turns[-1][1].append(text)
        else:
            turns.append((role, [text]))
    if not turns:
        return [], 'empty'
    if len(speakers) != 2:
        return [], 'single_speaker' if len(speakers) == 1 else 'more_than_two_speakers'
    return [(role, ' '.join(parts)) for role, parts in turns], None

def encode_pair(question, answer):
    prefix = ('화자1: ' + question + '\n화자2: ').encode('utf-8')
    reply = answer.encode('utf-8')
    tokens = np.empty(1 + len(prefix) + len(reply) + 1, dtype='<u2')
    tokens[0], tokens[-1] = 256, 257
    tokens[1:1+len(prefix)] = np.frombuffer(prefix, dtype=np.uint8)
    tokens[1+len(prefix):-1] = np.frombuffer(reply, dtype=np.uint8)
    return tokens, 1 + len(prefix)

data_spec = dict(version=2, zip_sha256=zip_sha256, text_field=DATA_CFG['text_field'],
                 renderer='first_seen_two_speakers_adjacent_1to2_merged_turns_v1',
                 tokenizer='utf8_byte_bos256_eos257_uint16le', loss='response_and_eos_only',
                 seed=CFG['seed'], validation_fraction=DATA_CFG['validation_fraction'])
assert 0 < DATA_CFG['validation_fraction'] < 1
data_key = hashlib.sha256(json.dumps(data_spec, sort_keys=True).encode()).hexdigest()[:16]
token_path = DATA_CACHE / f'{data_key}.uint16'
index_path = DATA_CACHE / f'{data_key}.json'


def prepare_pairs():
    documents, pairs, seen_ids = [], [], {}
    seen_docs, seen_pairs, skipped = set(), set(), {}
    offset = 0
    partial = token_path.with_suffix('.partial')
    with zipfile.ZipFile(local_zip) as archive, partial.open('wb') as output:
        members = sorted((info for info in archive.infolist()
                          if not info.is_dir() and info.filename.lower().endswith('.json')
                          and not info.filename.startswith('__MACOSX/')
                          and not info.filename.rsplit('/', 1)[-1].startswith('._')),
                         key=lambda info: info.filename)
        if not members:
            raise ValueError('ZIP 안에 JSON 파일이 없습니다.')
        for info in tqdm(members, desc='JSON → speaker1/speaker2 pairs'):
            with archive.open(info) as member:
                payload = json.loads(member.read().decode('utf-8-sig'))
            if not isinstance(payload, dict) or not isinstance(payload.get('document'), list):
                raise ValueError(f'{info.filename}: document list가 없습니다.')
            for number, document in enumerate(payload['document']):
                if not isinstance(document, dict):
                    raise ValueError(f'{info.filename}: document[{number}] must be an object')
                doc_id = str(document.get('id') or f'{info.filename}#{number}')
                turns, reason = dialogue_turns(document, f'{info.filename}:{doc_id}')
                if reason:
                    skipped[reason] = skipped.get(reason, 0) + 1
                    continue
                doc_hash = hashlib.sha256(json.dumps(turns, ensure_ascii=False).encode()).hexdigest()
                if doc_id in seen_ids and seen_ids[doc_id] != doc_hash:
                    raise ValueError(f'Duplicate document ID with different content: {doc_id}')
                seen_ids[doc_id] = doc_hash
                if doc_hash in seen_docs:
                    skipped['duplicate_document'] = skipped.get('duplicate_document', 0) + 1
                    continue
                seen_docs.add(doc_hash)
                count = 0
                for i in range(len(turns)-1):
                    if turns[i][0] != 1 or turns[i+1][0] != 2:
                        continue
                    question, answer = turns[i][1], turns[i+1][1]
                    pair_hash = hashlib.sha256(json.dumps([question, answer], ensure_ascii=False).encode()).hexdigest()
                    if pair_hash in seen_pairs:
                        skipped['duplicate_pair'] = skipped.get('duplicate_pair', 0) + 1
                        continue
                    seen_pairs.add(pair_hash)
                    tokens, prompt_tokens = encode_pair(question, answer)
                    tokens.tofile(output)
                    pairs.append(dict(document=len(documents), sha256=pair_hash, offset=offset,
                                      length=len(tokens), prompt_tokens=prompt_tokens))
                    offset += len(tokens)
                    count += 1
                if count:
                    documents.append(dict(id=doc_id, member=info.filename, sha256=doc_hash, pairs=count))
                else:
                    skipped['no_unique_pair'] = skipped.get('no_unique_pair', 0) + 1
    if len(documents) < 2:
        raise ValueError(f'유효한 2인 대화가 {len(documents)}개입니다. train/validation에 최소 2개 필요합니다. 제외: {skipped}')
    order = sorted(range(len(documents)), key=lambda i: hashlib.sha256(
        f'{CFG["seed"]}:{documents[i]["sha256"]}'.encode()).hexdigest())
    n_val = min(len(documents)-1, max(1, round(len(documents)*DATA_CFG['validation_fraction'])))
    validation = set(order[:n_val])
    for i, document in enumerate(documents):
        document['split'] = 'validation' if i in validation else 'train'
    for pair in pairs:
        pair['split'] = documents[pair['document']]['split']
    manifest = dict(spec=data_spec, documents=documents, pairs=pairs, total_tokens=offset,
                    json_files=len(members), skipped=skipped)
    partial.replace(token_path)
    index_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest

dataset_manifest = json.loads(index_path.read_text(encoding='utf-8')) if index_path.exists() else None
if (dataset_manifest is None or dataset_manifest['spec'] != data_spec or not token_path.exists()
        or token_path.stat().st_size != dataset_manifest['total_tokens']*2):
    dataset_manifest = prepare_pairs()
documents, pairs = dataset_manifest['documents'], dataset_manifest['pairs']
tokens_mm = np.memmap(token_path, mode='r', dtype='<u2')
train_ids = [i for i, row in enumerate(pairs) if row['split'] == 'train']
val_ids = [i for i, row in enumerate(pairs) if row['split'] == 'validation']
assert train_ids and val_ids
assert {pairs[i]['document'] for i in train_ids}.isdisjoint({pairs[i]['document'] for i in val_ids})
assert {pairs[i]['sha256'] for i in train_ids}.isdisjoint({pairs[i]['sha256'] for i in val_ids})
assert sum(row['length'] for row in pairs) == len(tokens_mm)
for row in pairs:
    start, end = row['offset'], row['offset'] + row['length']
    assert int(tokens_mm[start]) == 256 and int(tokens_mm[end-1]) == 257
    assert 1 < row['prompt_tokens'] < row['length']-1
data_stats = dict(json_files=dataset_manifest['json_files'], usable_documents=len(documents),
                  train_documents=len({pairs[i]['document'] for i in train_ids}),
                  validation_documents=len({pairs[i]['document'] for i in val_ids}),
                  train_pairs=len(train_ids), validation_pairs=len(val_ids),
                  train_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in train_ids),
                  validation_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in val_ids),
                  excluded=dataset_manifest['skipped'])
print(json.dumps(data_stats, ensure_ascii=False, indent=2))
RUN_DIR = ZIP_PATH.parent / 'flylm_runs' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
shutil.copyfile(index_path, RUN_DIR / 'dataset_manifest.json')
print('Drive 결과 저장 경로:', RUN_DIR)


## 2. 공식 원본 다운로드

MaleCNS v1.0 annotation, neuron neurotransmitter, connection Feather를 사용한다.
다운로드/길이 검사/SHA-256은 기존과 같다. 전체 annotation의 Traced neuron을 사용하고
그 뉴런들 사이에 존재하는 모든 edge/count를 보존한다. 생물학적 뉴런으로 확인되지 않은 orphan/glia/미할당 segment는 포함하지 않는다.


In [ ]:
BASE_URL = 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/'
SOURCES = {
    'annotations': ('body-annotations-male-cns-v1.0-minconf-0.5.feather', 14483314),
    'nt': ('body-neurotransmitters-male-cns-v1.0.feather', 43282834),
    'edges': ('connectome-weights-male-cns-v1.0-minconf-0.5.feather', 1051241946),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_file(name, expected_size):
    target = CACHE / name
    if not target.exists() or target.stat().st_size != expected_size:
        if shutil.disk_usage(CACHE).free < expected_size + 512 * 1024**2:
            raise RuntimeError('데이터 다운로드에 필요한 디스크 공간이 부족합니다.')
        partial = target.with_suffix(target.suffix + '.part')
        with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
            with partial.open('wb') as f, tqdm(total=expected_size, unit='B', unit_scale=True, desc=name) as bar:
                while True:
                    chunk = response.read(8 * 1024**2)
                    if not chunk:
                        break
                    f.write(chunk)
                    bar.update(len(chunk))
        if partial.stat().st_size != expected_size:
            raise RuntimeError(f'파일 크기 불일치: {name}. v1.0 파일/다운로드를 확인하세요.')
        partial.replace(target)
    return target

files = {key: download_file(*spec) for key, spec in SOURCES.items()}
source_manifest = {key: dict(url=BASE_URL + p.name, bytes=p.stat().st_size, sha256=sha256_file(p))
                   for key, p in files.items()}
print({key: entry['bytes'] for key, entry in source_manifest.items()})


## 3. 전체 뉴런, 전체 내부 연결의 CSR 캐시

Feather를 두 번 순회한다. 첫 순회는 destination별 edge 개수를 세고,
두 번째 순회는 디스크 memmap의 CSR 위치에 source/count를 채운다.
전체 원본 table을 pandas로 올리거나 edge 목록을 Python list에 누적하지 않는다.
중복 neuron pair는 count를 합산한다. CSR row=postsynaptic, column=presynaptic이다.
어떤 component도 버리지 않으며 self-edge도 원본에 있으면 유지한다.


In [ ]:
def require_columns(names, required, label):
    absent = set(required)-set(names)
    if absent:
        raise ValueError(f'{label}: missing {sorted(absent)}; found {list(names)}')

def map_ids(sorted_ids, values):
    positions = np.searchsorted(sorted_ids, values)
    safe = positions.clip(0, len(sorted_ids)-1)
    return positions, (positions < len(sorted_ids)) & (sorted_ids[safe] == values)

annotation = feather.read_table(files['annotations']).to_pandas()
require_columns(annotation.columns, ['bodyId','status','superclass'], 'annotations')
neurons = annotation.loc[annotation.status == 'Traced'].sort_values('bodyId').set_index('bodyId')
assert neurons.index.is_unique and not neurons.index.hasnans
body_ids = neurons.index.to_numpy(np.int64)
n = len(body_ids)
CFG['n_neurons'] = n
assert n > 100_000, f'전체 Traced 데이터인지 확인하세요: {n}'
print('Actual full Traced neuron count:', n)
print(neurons['superclass'].value_counts(dropna=False).to_string())

def full_edge_batches():
    with pa.memory_map(str(files['edges']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body_pre','body_post','weight'], 'edges')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            s, sm = map_ids(body_ids, batch.column('body_pre').to_numpy(zero_copy_only=False))
            d, dm = map_ids(body_ids, batch.column('body_post').to_numpy(zero_copy_only=False))
            keep = sm & dm
            if keep.any():
                w = batch.column('weight').to_numpy(zero_copy_only=False)[keep]
                if not np.isfinite(w).all() or (w <= 0).any():
                    raise ValueError('Invalid count')
                yield s[keep].astype(np.int32), d[keep].astype(np.int32), w.astype(np.int64)

extract_spec = dict(version=3, selection='all_status_Traced_no_component_filter',
                    orientation='post_row_pre_column', n=n,
                    source_sha256={k:v['sha256'] for k,v in source_manifest.items()})
cache_key = hashlib.sha256(json.dumps(extract_spec, sort_keys=True).encode()).hexdigest()[:16]
full_cache = CACHE / f'full_{cache_key}'
full_cache.mkdir(exist_ok=True)
manifest_path = full_cache / 'complete.json'
if not manifest_path.exists():
    degree = np.zeros(n, np.int64)
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 1: full degrees'):
        degree += np.bincount(d, minlength=n)
    raw_edges = int(degree.sum())
    if raw_edges == 0:
        raise ValueError('No full-neuron edges')
    required_disk = raw_edges*24 + 512*1024**2
    if shutil.disk_usage(full_cache).free < required_disk:
        raise RuntimeError(f'CSR 구축 디스크가 부족합니다. 약 {required_disk/2**30:.1f} GiB 여유가 필요합니다.')
    crow = np.concatenate(([0], np.cumsum(degree)))
    col = np.lib.format.open_memmap(full_cache/'src.npy', mode='w+', dtype=np.int32, shape=(raw_edges,))
    count = np.lib.format.open_memmap(full_cache/'count.npy', mode='w+', dtype=np.int64, shape=(raw_edges,))
    cursor = crow[:-1].copy()
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 2: write full CSR'):
        order = np.argsort(d, kind='stable')
        sd = d[order]
        starts = np.r_[0, np.flatnonzero(sd[1:] != sd[:-1])+1]
        sizes = np.diff(np.r_[starts, len(sd)])
        local = np.arange(len(sd))-np.repeat(starts, sizes)
        positions = cursor[sd]+local
        col[positions], count[positions] = s[order], w[order]
        cursor += np.bincount(d, minlength=n)
    assert np.array_equal(cursor, crow[1:])
    col.flush()
    count.flush()
    full = sparse.csr_matrix((count, col, crow), shape=(n,n), copy=False)
    full.sum_duplicates()
    full.sort_indices()
    nnz = int(full.nnz)
    # sum_duplicates may shrink arrays/reallocate; persist the canonical prefix explicitly.
    col[:nnz], count[:nnz] = full.indices, full.data
    col.flush()
    count.flush()
    np.save(full_cache/'crow.npy', full.indptr.astype(np.int64))
    np.save(full_cache/'body_ids.npy', body_ids)
    manifest_path.write_text(json.dumps(dict(spec=extract_spec, raw_edges=raw_edges, nnz=nnz)), encoding='utf-8')
    del full, col, count, cursor, degree, crow
    gc.collect()
full_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert full_manifest['spec'] == extract_spec
crow = np.load(full_cache/'crow.npy', mmap_mode='r')
src = np.load(full_cache/'src.npy', mmap_mode='r')[:full_manifest['nnz']]
counts = np.load(full_cache/'count.npy', mmap_mode='r')[:full_manifest['nnz']]
assert int(crow[-1]) == len(src) == len(counts)
assert np.array_equal(body_ids, np.load(full_cache/'body_ids.npy'))
graph_data = dict(body_ids=body_ids, crow=crow, src=src, counts=counts)
print(dict(neurons=n, edges=len(src), retained='all Traced-to-Traced pairs'))


## 4. 실제 sensory/motor 포트와 세포 유형

전체 Traced 집합을 유지한다. 경로가 없는 뉴런도 삭제하지 않는다.
신경전달물질은 ground truth 우선, 없으면 confidence ≥ 0.8의 예측을 사용한다.
ACh 양수 / GABA 음수는 이 모델의 근사다. 수용체 정보가 없는 다른 전달물질은 부호를 학습한다.
도파민 등을 이름만으로 보상 학습 규칙으로 해석하지 않는다.
`type` annotation이 있는 뉴런은 같은 유형끼리 동역학 매개변수를 공유하고,
미분류 뉴런은 `class`, `superclass` 순으로 묶는다. 이 fallback도 저장한다.

In [ ]:
KNOWN_NT = {'acetylcholine', 'gaba', 'glutamate', 'histamine', 'dopamine', 'serotonin', 'octopamine'}

def attach_nt(body_ids):
    selected_parts = []
    wanted = np.sort(body_ids)
    with pa.memory_map(str(files['nt']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body', 'ground_truth', 'predicted_nt',
                                              'predicted_nt_confidence', 'consensus_nt'], 'neurotransmitters')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            _, keep = map_ids(wanted, batch.column('body').to_numpy(zero_copy_only=False))
            if keep.any():
                selected_parts.append(batch.filter(pa.array(keep)).to_pandas())
    if not selected_parts:
        raise RuntimeError('선택된 body와 전달물질 테이블의 join이 비어 있습니다.')
    table = pd.concat(selected_parts, ignore_index=True)
    if table['body'].duplicated().any():
        raise ValueError('Duplicate NT body IDs: explicit aggregation is required')
    table = table.set_index('body').reindex(body_ids)
    truth = table['ground_truth'].fillna('').str.strip().str.lower()
    predicted = table['predicted_nt'].fillna('').str.strip().str.lower()
    confidence = pd.to_numeric(table['predicted_nt_confidence'], errors='coerce')
    trusted = truth.isin(KNOWN_NT)
    confident = predicted.isin(KNOWN_NT) & confidence.ge(CFG['nt_confidence']) & confidence.le(1)
    effective = np.where(trusted, truth, np.where(confident, predicted, 'unknown'))
    origin = np.where(trusted, 'ground_truth', np.where(confident, 'predicted_nt', 'unknown'))
    table['effective_nt'] = effective
    table['effective_source'] = origin
    table.index.name = 'body'
    return table


SENSORY_CLASSES = ['cb_sensory','ol_sensory','vnc_sensory','sensory_ascending','sensory_descending']
MOTOR_CLASSES = ['cb_motor','vnc_motor']
input_nodes = np.flatnonzero(neurons.superclass.isin(SENSORY_CLASSES)).astype(np.int64)
output_nodes = np.flatnonzero(neurons.superclass.isin(MOTOR_CLASSES)).astype(np.int64)
assert len(input_nodes) and len(output_nodes)
assert len(np.intersect1d(input_nodes, output_nodes)) == 0
nt_table = attach_nt(body_ids)
node_sign = nt_table.effective_nt.map({'acetylcholine':1.0,'gaba':-1.0}).fillna(0).to_numpy(np.float32)
if not CFG['use_nt_signs']:
    node_sign[:] = 0
edge_sign = node_sign[src]
from scipy.sparse.csgraph import dijkstra
# Multi-source min_only returns a length-N vector, never an inputs×N dense distance matrix.
adj_post_pre = sparse.csr_matrix((np.ones(len(src), np.uint8), src, crow), shape=(n,n), copy=False)
adj_pre_post = adj_post_pre.T.tocsr()
from_sensory = dijkstra(adj_pre_post, directed=True, indices=input_nodes, min_only=True, unweighted=True)
to_motor = dijkstra(adj_post_pre, directed=True, indices=output_nodes, min_only=True, unweighted=True)
graph_stats = dict(neurons=n, edges=len(src), sensory_neurons=len(input_nodes), motor_neurons=len(output_nodes),
                   reachable_from_sensory=int(np.isfinite(from_sensory).sum()),
                   can_reach_motor=int(np.isfinite(to_motor).sum()),
                   sensory_to_motor_path_neurons=int((np.isfinite(from_sensory)&np.isfinite(to_motor)).sum()),
                   reachable_motors=int(np.isfinite(from_sensory[output_nodes]).sum()),
                   positive_edges=int((edge_sign>0).sum()), negative_edges=int((edge_sign<0).sum()),
                   unconstrained_edges=int((edge_sign==0).sum()))
assert graph_stats['reachable_motors'] > 0, 'No biological sensory→motor path; annotations/direction를 확인하세요.'
print(graph_stats)
print(nt_table.groupby(['effective_source','effective_nt']).size().to_string())
neurons[['superclass','class']].assign(is_input=neurons.index.isin(body_ids[input_nodes]),
               is_output=neurons.index.isin(body_ids[output_nodes])).to_csv(RUN_DIR/'neuron_ports.csv')
nt_table.to_csv(RUN_DIR/'neurotransmitters.csv')
del adj_post_pre, adj_pre_post, annotation
gc.collect()


def dynamics_types(frame):
    labels = pd.Series('', index=frame.index, dtype=object)
    for column in ['type', 'class', 'superclass']:
        if column not in frame:
            continue
        values = frame[column].fillna('').astype(str).str.strip()
        valid = ~values.str.lower().isin(['', 'none', 'nan', 'unknown', 'tbc', 'untyped'])
        use = labels.eq('') & valid
        labels.loc[use] = column + ':' + values.loc[use]
    labels = labels.mask(labels.eq(''), 'unknown')
    codes, names = pd.factorize(labels, sort=True)
    return codes.astype(np.int64), list(names)

type_ids, type_names = dynamics_types(neurons)
print('Dynamics groups:', len(type_names), 'type-annotated neurons:',
      sum(type_names[i].startswith('type:') for i in type_ids))

## 5. 유형별 동역학과 내부 상태

각 tick에서 이전 `s`가 연결망을 통해 전류를 전달하고, 막 상태가 이를 적분한다.
`r = relu(tanh(v − threshold))`는 0~1의 신호 방출량이다.
`s`는 빠르게, `a`는 느리게 `r`을 추적하며 `a`는 막 입력을 억제한다.
이는 스파이크를 직접 구현하지 않는 연속 신호 모델이다.

유형별 시간 상수는 동일한 초기값에서 시작해 개별적으로 학습한다. 초기 상태는 모두 0이다.
기존 시냅스 수의 log 정규화 prior를 사용하고, 알려진 부호의 연결은 양의 배율로만 조절한다.
임의 부호 연결은 bounded tanh로 조절한다. 이 가중치는 실제 전도도 측정값이 아니다.

In [ ]:
BOS, EOS, VOCAB = 256, 257, 258

def encode(text):
    return [BOS] + list(text.encode('utf-8')) + [EOS]

def decode(ids):
    return bytes(int(i) for i in ids if 0 <= int(i) < 256).decode('utf-8', errors='replace')

class SparseEdges(torch.autograd.Function):
    @staticmethod
    def forward(ctx, h, weights, rowptr, col, dst, block):
        matrix = torch.sparse_csr_tensor(rowptr, col, weights, size=(h.shape[1], h.shape[1]),
                                         device=h.device, dtype=h.dtype)
        ctx.save_for_backward(h, weights, rowptr, col, dst)
        ctx.block = block
        return torch.sparse.mm(matrix, h.T.contiguous()).T.contiguous()

    @staticmethod
    def backward(ctx, grad_output):
        h, weights, rowptr, col, dst = ctx.saved_tensors
        grad_h = torch.zeros_like(h) if ctx.needs_input_grad[0] else None
        grad_w = torch.empty_like(weights) if ctx.needs_input_grad[1] else None
        for lo in range(0, col.numel(), ctx.block):
            hi = min(col.numel(), lo+ctx.block)
            sources, targets = col[lo:hi], dst[lo:hi]
            g = grad_output.index_select(1, targets)
            if grad_w is not None:
                grad_w[lo:hi] = (g*h.index_select(1, sources)).sum(0)
            if grad_h is not None:
                grad_h.index_add_(1, sources, g*weights[lo:hi])
        return grad_h, grad_w, None, None, None, None

class ConnectomeLM(nn.Module):
    def __init__(self, cfg, rowptr, src, counts, edge_sign, inputs, outputs, type_ids, device):
        super().__init__()
        self.n, self.k = cfg['n_neurons'], cfg['brain_steps']
        self.block, self.checkpoint_tokens = cfg['edge_block'], cfg['checkpoint_tokens']
        for name, value in [('rowptr', rowptr), ('src', src), ('inputs', inputs),
                            ('outputs', outputs), ('type_ids', type_ids)]:
            self.register_buffer(name, torch.tensor(np.asarray(value), dtype=torch.long,
                                                   device=device), persistent=False)
        assert len(type_ids) == self.n and min(type_ids) >= 0
        self.register_buffer('dst', torch.repeat_interleave(
            torch.arange(self.n, device=device), self.rowptr.diff()), persistent=False)
        sign = torch.tensor(np.asarray(edge_sign), dtype=torch.float32, device=device)
        self.register_buffer('edge_sign', sign, persistent=False)
        c = torch.tensor(np.asarray(counts), dtype=torch.float32, device=device).log1p()
        incoming = torch.zeros(self.n, device=device).index_add_(0, self.dst, c)
        self.register_buffer('prior', c / incoming[self.dst].clamp_min(1e-8), persistent=False)
        raw = torch.where(sign != 0, torch.full_like(sign, math.log(0.2 / 0.8)),
                          torch.where(torch.rand_like(sign) < 0.5, -0.2554128, 0.2554128))
        self.theta = nn.Parameter(raw)
        ntypes = int(max(type_ids)) + 1
        low, high, initial = [torch.tensor(cfg[key], device=device)[:, None]
                              for key in ['tau_min', 'tau_max', 'tau_initial']]
        assert torch.all((low > 0) & (initial > low) & (initial < high))
        self.register_buffer('tau_low', low)
        self.register_buffer('tau_high', high)
        fraction = (initial - low) / (high - low)
        self.raw_tau = nn.Parameter(torch.logit(fraction).expand(3, ntypes).clone())
        self.threshold = nn.Parameter(torch.zeros(ntypes, device=device))
        self.raw_adaptation = nn.Parameter(torch.full((ntypes,), math.log(0.1 / 0.9), device=device))
        d = cfg['embedding_dim']
        self.embedding = nn.Embedding(VOCAB, d, device=device)
        self.input_adapter = nn.Linear(d, len(inputs), bias=False, device=device)
        self.motor_norm = nn.LayerNorm(len(outputs), eps=1e-6, elementwise_affine=False, device=device)
        self.output_adapter = nn.Linear(len(outputs), d, bias=False, device=device)

    def effective_weights(self):
        magnitude = 0.25 + 3.75 * self.theta.sigmoid()
        signed = torch.where(self.edge_sign != 0, self.edge_sign * magnitude,
                             4 * self.theta.tanh())
        return self.prior * signed

    def dynamics(self):
        tau = self.tau_low + (self.tau_high - self.tau_low) * self.raw_tau.sigmoid()
        alpha = -torch.expm1(-1 / tau[:, self.type_ids])
        threshold = 0.5 * self.threshold.tanh()[self.type_ids]
        adaptation = self.raw_adaptation.sigmoid()[self.type_ids]
        return alpha, threshold, adaptation

    def initial_state(self, batch):
        return self.embedding.weight.new_zeros(3, batch, self.n)

    def advance(self, token, state, weights, alpha, threshold, adaptation):
        v, signal, slow = state.unbind(0)
        value = self.input_adapter(self.embedding(token))
        drive = torch.zeros_like(v).index_add(1, self.inputs, value)
        for _ in range(self.k):
            message = SparseEdges.apply(signal, weights, self.rowptr, self.src, self.dst, self.block)
            v = v + alpha[0] * (-v + message + drive - adaptation * slow)
            rate = torch.relu(torch.tanh(v - threshold))
            signal = signal + alpha[1] * (rate - signal)
            slow = slow + alpha[2] * (rate - slow)
        return torch.stack((v, signal, slow))

    def forward(self, tokens, state=None):
        if tokens.ndim != 2 or tokens.shape[1] == 0:
            raise ValueError('Nonempty tokens [B,T] required')
        state = self.initial_state(tokens.shape[0]) if state is None else state
        if state.shape != (3, tokens.shape[0], self.n):
            raise ValueError('State must be [3,B,N]: membrane, synaptic signal, adaptation')
        weights = self.effective_weights()
        alpha, threshold, adaptation = self.dynamics()
        logits = []
        for t in range(tokens.shape[1]):
            args = (tokens[:, t], state, weights, alpha, threshold, adaptation)
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                state = checkpoint(self.advance, *args, use_reentrant=False, preserve_rng_state=False)
            else:
                state = self.advance(*args)
            motor = self.motor_norm(state[1].index_select(1, self.outputs))
            logits.append(F.linear(self.output_adapter(motor), self.embedding.weight))
        return torch.stack(logits, dim=1), state

def make_model():
    torch.manual_seed(CFG['seed'])
    torch.cuda.manual_seed_all(CFG['seed'])
    return ConnectomeLM(CFG, crow, src, counts, edge_sign, input_nodes, output_nodes, type_ids, device)

# ponytail: single compartment per neuron; add anatomical compartments only when needed.
# Planning estimate only; the next cell measures an actual backward + Adam step.
estimated = len(src)*88 + n*CFG['tbptt']*3*4*8 + 2*1024**3
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(dict(edges=len(src), planning_GiB=estimated/2**30, free_GPU_GiB=free_bytes/2**30))
if estimated > free_bytes*0.9:
    raise RuntimeError('전체 그래프/TBPTT64 메모리 여유가 부족할 수 있습니다. 더 큰 GPU를 사용하세요. 자동 축소하지 않습니다.')
model = make_model()
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))

## 6. 실행 전 검증

작은 그래프에서 sparse 값/기울기, checkpoint, 분할 실행, 입력 인과성, 내부 상태 전달을 검사한다.
이어서 **실제 전체 그래프에서 64-token backward와 Adam 1회**를 실행해 peak memory를 측정한다.
검증 모델은 폐기하고 학습은 같은 seed로 새로 시작한다. 아래 PASS는 대화 능력 검증이 아니다.

In [ ]:
def numerical_checks():
    cfg = dict(CFG, n_neurons=6, embedding_dim=8, edge_block=3)
    row = np.array([0, 1, 2, 3, 4, 5, 6])
    source = np.array([5, 0, 1, 2, 3, 4])
    sign = np.array([1, 1, 1, -1, 0, 1], dtype=np.float32)
    m = ConnectomeLM(cfg, row, source, np.ones(6), sign, [0, 1], [2, 3, 4],
                    np.array([0, 0, 1, 1, 2, 2]), device)
    h = torch.randn(2, 6, device=device, requires_grad=True)
    w = torch.randn(6, device=device, requires_grad=True)
    actual = SparseEdges.apply(h, w, m.rowptr, m.src, m.dst, 3)
    dense = torch.zeros(6, 6, device=device).index_put((m.dst, m.src), w)
    expected = h @ dense.T
    torch.testing.assert_close(actual, expected)
    upstream = torch.randn_like(actual)
    got = torch.autograd.grad(actual, (h, w), upstream)
    want = torch.autograd.grad(expected, (h, w), upstream)
    for a, b in zip(got, want):
        torch.testing.assert_close(a, b)
    x = torch.arange(16, device=device).unsqueeze(0)
    gradients = []
    for use_checkpoint in (False, True):
        m.checkpoint_tokens = use_checkpoint
        m.zero_grad(set_to_none=True)
        logits, state = m(x)
        F.cross_entropy(logits.flatten(0, 1), x.flatten()).backward()
        gradients.append([p.grad.clone() for p in m.parameters()])
    for a, b in zip(*gradients):
        torch.testing.assert_close(a, b, rtol=1e-5, atol=1e-7)
    with torch.no_grad():
        logits, end = m(x)
        first, state = m(x[:, :7])
        second, split_end = m(x[:, 7:], state)
        torch.testing.assert_close(logits, torch.cat((first, second), 1))
        torch.testing.assert_close(end, split_end)
        changed = x.clone()
        changed[:, 7:] += 1
        other, _ = m(changed)
        torch.testing.assert_close(logits[:, :7], other[:, :7])
        assert (end[1:] >= 0).all() and (end[1:] <= 1).all()
        assert torch.isfinite(end).all()
        continuation = torch.tensor([[20, 21]], device=device)
        remembered, _ = m(continuation, state)
        reset, _ = m(continuation)
        assert (remembered - reset).abs().max() > 1e-7
        weights = m.effective_weights()
        assert (weights[m.edge_sign > 0] > 0).all()
        assert (weights[m.edge_sign < 0] < 0).all()
    print('PASS: sparse gradients, checkpoint, streaming, causality, internal state, NT signs')

def full_backward_check(m):
    torch.cuda.reset_peak_memory_stats()
    x = (torch.arange(64, device=device) % 256).unsqueeze(0)
    logits, state = m(x)
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, VOCAB), x[:, 1:].reshape(-1))
    loss.backward()
    for name, parameter in m.named_parameters():
        assert parameter.grad is not None and torch.isfinite(parameter.grad).all(), name
    input_gradient = m.input_adapter.weight.grad.norm().item()
    assert input_gradient > 0, 'sensory→motor gradient가 0입니다. 학습을 중단합니다.'
    edge_coverage = (m.theta.grad != 0).float().mean().item()
    assert edge_coverage > 0
    nn.utils.clip_grad_norm_(m.parameters(), 1.0, error_if_nonfinite=True)
    probe_optimizer = torch.optim.Adam(m.parameters(), lr=CFG['lr'], foreach=False)
    probe_optimizer.step()
    assert all(torch.isfinite(p).all() for p in m.parameters())
    torch.cuda.synchronize()
    print(dict(full_tbptt=64, loss=float(loss.detach()), input_gradient=input_gradient,
               edge_gradient_fraction=edge_coverage,
               peak_allocated_GiB=torch.cuda.max_memory_allocated()/2**30,
               peak_reserved_GiB=torch.cuda.max_memory_reserved()/2**30))

numerical_checks()
full_backward_check(model)
del model
gc.collect()
torch.cuda.empty_cache()

## 7. 내부 상태만 전달하는 response-only TBPTT64

첫 답변 구간에 마지막 최대 32 prompt bytes가 포함되도록 경계를 맞춘다.
앞부분 prompt는 순차적으로 상태에 반영하고, 첫 답변 loss는 마지막 prompt 부분까지 역전파된다.
이전 64단계 밖에는 직접적인 gradient가 도달하지 않는다. 장문 기억 학습의 한계는 남는다.
질문은 한 번만 읽는다. 모든 이후 구간에는 직전 뉴런 상태와 현재 byte만 전달한다.
문서 단위 validation은 정답 답변의 다음 byte loss/accuracy를 측정한다.
체크포인트는 추론/추가 학습용 가중치이며 optimizer와 데이터 cursor의 정확한 재개는 제공하지 않는다.

In [ ]:
IGNORE_INDEX = -100
assert (BOS, EOS) == (256, 257)

def response_mask(start, length, prompt_tokens):
    # x[k]가 예측하는 y[k]의 원래 sequence 위치는 start+k+1이다.
    return np.arange(start+1, start+length+1) >= prompt_tokens

def chunk_spans(length, prompt_tokens):
    warmup = max(0, prompt_tokens - 1 - CFG['prompt_grad_tokens'])
    for start in range(0, warmup, CFG['tbptt']):
        yield start, min(CFG['tbptt'], warmup - start)
    for start in range(warmup, length - 1, CFG['tbptt']):
        yield start, min(CFG['tbptt'], length - 1 - start)

def pair_chunks(pair_index, max_response_tokens=None):
    row = pairs[pair_index]
    stop = row['length']
    if max_response_tokens is not None:
        stop = min(stop, row['prompt_tokens'] + max_response_tokens)
    for start, length in chunk_spans(stop, row['prompt_tokens']):
        offset = row['offset'] + start
        window = torch.from_numpy(np.array(tokens_mm[offset:offset+length+1], dtype=np.int64)).unsqueeze(0).to(device)
        labels = window[:, 1:].clone()
        mask = torch.from_numpy(response_mask(start, length, row['prompt_tokens'])).to(device).unsqueeze(0)
        labels.masked_fill_(~mask, IGNORE_INDEX)
        yield window[:, :-1], labels, int(mask.sum())

def check_response_format():
    fixture = dict(utterance=[
        dict(speaker_id='A', form='질문 하나'), dict(speaker_id='A', form='질문 둘'),
        dict(speaker_id='B', form='답변 하나'), dict(speaker_id='A', form='다음 질문'),
        dict(speaker_id='B', form='다음 답변')])
    # text_field 설정과 무관하게 동일 구조를 검사한다.
    for utterance in fixture['utterance']:
        utterance[DATA_CFG['text_field']] = utterance['form']
    turns, reason = dialogue_turns(fixture, 'fixture')
    assert reason is None and turns == [(1, '질문 하나 질문 둘'), (2, '답변 하나'), (1, '다음 질문'), (2, '다음 답변')]
    assert dialogue_turns(dict(utterance=fixture['utterance'][:2]), 'monologue')[1] == 'single_speaker'
    encoded, boundary = encode_pair('질문', '답변')
    actual_targets = []
    for start, length in chunk_spans(len(encoded), boundary):
        mask = response_mask(start, length, boundary)
        actual_targets.extend(encoded[start+1:start+length+1][mask].tolist())
    assert actual_targets == list('답변'.encode('utf-8')) + [EOS]
    assert not response_mask(0, boundary-1, boundary).any()
    assert response_mask(boundary-1, 1, boundary)[0]  # 첫 답변 byte가 loss에 포함됨
    print('PASS: speaker mapping, turn merge, monologue exclusion, response-only mask + EOS')

check_response_format()
eval_ids = sorted(val_ids, key=lambda i: pairs[i]['sha256'])[:CFG['eval_max_pairs']]


@torch.no_grad()
def evaluate(m):
    was_training = m.training
    m.eval()
    loss_sum, correct, total = 0., 0, 0
    try:
        for pair_index in eval_ids:
            state = None
            for x, labels, supervised in pair_chunks(pair_index, CFG['eval_response_tokens']):
                logits, state = m(x, state)
                if supervised:
                    loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
                                                ignore_index=IGNORE_INDEX, reduction='sum').item()
                    correct += int(((logits.argmax(-1) == labels) & (labels != IGNORE_INDEX)).sum())
                    total += supervised
    finally:
        m.train(was_training)
    mean = loss_sum / total
    if not math.isfinite(mean):
        raise RuntimeError('Nonfinite validation loss')
    return dict(response_loss=mean, response_perplexity=math.exp(mean) if mean < 700 else None,
                response_accuracy=correct/total, response_tokens=total, pairs=len(eval_ids),
                full_validation=total == data_stats['validation_response_tokens'])

def to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(to_cpu(item) for item in value)
    return value


# Immutable full graph is exported once, avoiding duplicate graph copies in each checkpoint.
graph_export = RUN_DIR/'graph'
graph_export.mkdir(exist_ok=True)
for filename in ['src.npy','count.npy','crow.npy','body_ids.npy','complete.json']:
    if not (graph_export/filename).exists():
        shutil.copyfile(full_cache/filename,graph_export/filename)
np.save(graph_export/'edge_sign.npy',edge_sign)
np.save(graph_export/'inputs.npy',input_nodes)
np.save(graph_export/'outputs.npy',output_nodes)
np.save(graph_export/'type_ids.npy',type_ids)
(graph_export/'type_names.json').write_text(json.dumps(type_names, ensure_ascii=False), encoding='utf-8')

def save_checkpoint(name,m,optimizer,updates,validation):
    # Parameter-only full-graph checkpoint: optimizer/data cursor are not saved.
    payload = dict(architecture='malecns_dynamics_v1', config=CFG,model_state=to_cpu(m.state_dict()),graph_directory='graph',
                   extract_spec=extract_spec,sources=source_manifest,data_spec=data_spec,
                   data_config=DATA_CFG,updates=updates,validation=validation)
    local = CACHE/name
    torch.save(payload,local)
    temporary = RUN_DIR/(name+'.partial')
    shutil.copyfile(local,temporary)
    temporary.replace(RUN_DIR/name)
    del payload

def save_history(rows):
    (RUN_DIR / 'history.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
model = make_model()  # 이 셀 재실행은 새 학습이다.
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], foreach=False)
initial = dict(evaluate(model), evaluated_at_update=0)
best_loss, last_validation = initial['response_loss'], initial
history = [dict(update=0, validation=initial)]
save_checkpoint('best.pt', model, optimizer, 0, initial)
save_history(history)
updates, seen_response_tokens, processed_tokens, completed_epochs = 0, 0, 0, 0
pairs_started, pairs_completed = 0, 0
training_seconds, window_loss, window_tokens = 0.0, 0.0, 0

def supervised_updates(row):
    warmup = max(0, row['prompt_tokens'] - 1 - CFG['prompt_grad_tokens'])
    return math.ceil((row['length'] - 1 - warmup) / CFG['tbptt'])

updates_per_epoch = sum(supervised_updates(pairs[i]) for i in train_ids)
planned_updates = updates_per_epoch * CFG['epochs']
if CFG['train_updates'] is not None:
    planned_updates = min(planned_updates, CFG['train_updates'])
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
wall_start = time.perf_counter()
with tqdm(total=planned_updates, desc='Response train updates') as progress:
    for epoch in range(CFG['epochs']):
        order = train_ids.copy()
        random.Random(CFG['seed'] + epoch).shuffle(order)
        finished_in_epoch = 0
        for pair_index in order:
            if updates >= planned_updates:
                break
            state, pair_response_seen = None, 0
            pairs_started += 1
            for x, labels, supervised in pair_chunks(pair_index):
                if updates >= planned_updates:
                    break
                model.train()
                torch.cuda.synchronize()
                tick = time.perf_counter()
                if supervised == 0:
                    # 입력은 recurrent state에 반영하되 입력 자체를 정답으로 학습하지 않는다.
                    with torch.no_grad():
                        _, state = model(x, state)
                    state = state.detach()
                    torch.cuda.synchronize()
                    training_seconds += time.perf_counter() - tick
                    processed_tokens += x.numel()
                    continue
                optimizer.zero_grad(set_to_none=True)
                logits, next_state = model(x, state)
                loss = F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1), ignore_index=IGNORE_INDEX)
                if not torch.isfinite(loss):
                    raise RuntimeError(f'Nonfinite response loss before update {updates+1}')
                loss.backward()
                grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
                state = next_state.detach()
                torch.cuda.synchronize()
                training_seconds += time.perf_counter() - tick
                updates += 1
                processed_tokens += x.numel()
                seen_response_tokens += supervised
                pair_response_seen += supervised
                window_loss += loss.item() * supervised
                window_tokens += supervised
                progress.update(1)
                if updates == 1 or updates % CFG['log_every'] == 0:
                    row = dict(update=updates, epoch=epoch+1, response_tokens=seen_response_tokens,
                               train_response_loss=window_loss/window_tokens, grad_norm=float(grad_norm))
                    history.append(row)
                    tqdm.write(json.dumps(row))
                    window_loss, window_tokens = 0.0, 0
                if updates % CFG['eval_every'] == 0:
                    last_validation = dict(evaluate(model), evaluated_at_update=updates)
                    history.append(dict(update=updates, validation=last_validation))
                    tqdm.write(json.dumps(dict(update=updates, validation=last_validation)))
                    if last_validation['response_loss'] < best_loss:
                        best_loss = last_validation['response_loss']
                        save_checkpoint('best.pt', model, optimizer, updates, last_validation)
                if updates % CFG['save_every'] == 0:
                    save_checkpoint('latest.pt', model, optimizer, updates, last_validation)
                    save_history(history)
            pair = pairs[pair_index]
            if pair_response_seen == pair['length']-pair['prompt_tokens']:
                pairs_completed += 1
                finished_in_epoch += 1
        if finished_in_epoch == len(train_ids):
            completed_epochs += 1
        if updates >= planned_updates:
            break
final = dict(evaluate(model), evaluated_at_update=updates)
history.append(dict(update=updates, validation=final, final=True))
if final['response_loss'] < best_loss:
    best_loss = final['response_loss']
    save_checkpoint('best.pt', model, optimizer, updates, final)
save_checkpoint('latest.pt', model, optimizer, updates, final)
save_history(history)
torch.cuda.synchronize()
report = dict(initial_validation=initial, final_validation=final, best_response_loss=best_loss,
              updates=updates, train_response_tokens=seen_response_tokens, processed_model_tokens=processed_tokens,
              pairs_started=pairs_started, pairs_completed=pairs_completed, completed_epochs=completed_epochs,
              stop_reason='epochs_completed' if completed_epochs == CFG['epochs'] else 'update_limit',
              training_seconds=training_seconds, response_tokens_per_second=seen_response_tokens/training_seconds,
              wall_seconds=time.perf_counter()-wall_start,
              peak_allocated_MiB=torch.cuda.max_memory_allocated()/2**20,
              peak_reserved_MiB=torch.cuda.max_memory_reserved()/2**20)
print(json.dumps(report, ensure_ascii=False, indent=2))

# Release the final autograd graph before restoring another full model.
del logits, next_state, loss, grad_norm, state

## 8. 생성과 가중치 복원

prompt를 한 번 읽고, 그 다음에는 생성한 byte와 뉴런 상태만 전달한다.
UTF-8의 유효한 다음 byte만 허용한다. temperature=0은 greedy, 양수는 sampling이다.
이 제약은 인코딩만 보장하며 문장 품질을 보장하지 않는다.
실행 예제는 학습 결과를 그대로 출력한다. best checkpoint는 모델을 하나만 유지하면서 복원한다.

In [ ]:
import codecs

def inference_inputs(question):
    question = ' '.join(question.split())
    if not question:
        raise ValueError('Nonempty question required')
    return [BOS] + list(('화자1: ' + question + '\n화자2: ').encode('utf-8'))

@torch.no_grad()
def prefill_question(m, question):
    prefix = inference_inputs(question)
    state = None
    for start in range(0, len(prefix), CFG['tbptt']):
        ids = torch.tensor([prefix[start:start+CFG['tbptt']]], device=m.embedding.weight.device)
        logits, state = m(ids, state)
    return logits, state

def valid_utf8_tokens(decoder):
    state = decoder.getstate()
    probe = codecs.getincrementaldecoder('utf-8')('strict')
    allowed = []
    for value in range(256):
        probe.setstate(state)
        try:
            probe.decode(bytes([value]), final=False)
            allowed.append(value)
        except UnicodeDecodeError:
            pass
    if not state[0]:
        allowed.append(EOS)
    return allowed

@torch.no_grad()
def reply(m, question, max_new_bytes=180, temperature=0.8):
    if max_new_bytes < 1 or temperature < 0:
        raise ValueError('max_new_bytes >= 1, temperature >= 0 required')
    was_training = m.training
    m.eval()
    decoder = codecs.getincrementaldecoder('utf-8')('strict')
    pieces = []
    try:
        logits, state = prefill_question(m, question)
        for _ in range(max_new_bytes):
            allowed = torch.tensor(valid_utf8_tokens(decoder), device=logits.device)
            scores = logits[0, -1].index_select(0, allowed)
            if not torch.isfinite(scores).all():
                raise RuntimeError('Nonfinite generation logits')
            selected = scores.argmax() if temperature == 0 else torch.multinomial(
                (scores / temperature).softmax(-1), 1)[0]
            token = int(allowed[selected])
            if token == EOS:
                break
            pieces.append(decoder.decode(bytes([token]), final=False))
            logits, state = m(torch.tensor([[token]], device=logits.device), state)
        # A partial UTF-8 character at the byte budget is omitted, never replaced with U+FFFD.
        return ''.join(pieces)
    finally:
        m.train(was_training)

def load_full_checkpoint(checkpoint_path):
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if saved.get('architecture') != 'malecns_dynamics_v1':
        raise ValueError('이 노트북의 dynamics checkpoint만 복원할 수 있습니다.')
    folder = Path(checkpoint_path).parent / saved['graph_directory']
    info = json.loads((folder/'complete.json').read_text())
    count = info['nnz']
    restored = ConnectomeLM(saved['config'], np.load(folder/'crow.npy', mmap_mode='r'),
        np.load(folder/'src.npy', mmap_mode='r')[:count],
        np.load(folder/'count.npy', mmap_mode='r')[:count],
        np.load(folder/'edge_sign.npy', mmap_mode='r'),
        np.load(folder/'inputs.npy'), np.load(folder/'outputs.npy'),
        np.load(folder/'type_ids.npy'), device)
    restored.load_state_dict(saved['model_state'])
    return restored.eval()

# Free Adam and current model before loading the best full graph model.
del optimizer, model
gc.collect()
torch.cuda.empty_cache()
model = load_full_checkpoint(RUN_DIR/'best.pt')
examples = [dict(input=q, output=reply(model, q)) for q in
            ['안녕하세요?', '좋아하는 음식은 뭐예요?', '기차는 어디에서 타나요?']]
print(json.dumps(examples, ensure_ascii=False, indent=2))
(RUN_DIR/'examples.json').write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding='utf-8')
(RUN_DIR/'result.json').write_text(json.dumps(dict(config=CFG, versions=VERSIONS,
    sources=source_manifest, data_config=DATA_CFG, data_spec=data_spec, data_stats=data_stats,
    graph=graph_stats, dynamics_groups=len(type_names), metrics=report),
    ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', RUN_DIR)

# print(reply(model, '오늘 기분이 어때요?', temperature=0))

## 실행 결과 확인

- 전체 그래프 수, sensory/motor 수, 동역학 유형 수가 출력된다.
- 기본 수치 검증과 전체 64-token backward/Adam 검사를 통과해야 학습이 시작된다.
- validation loss와 실제 생성 결과로 학습 진행을 판단한다. 1,000 update는 완성된 대화를 보장하지 않는다.
- Drive의 실행별 디렉터리에 graph, 유형 매핑, best/latest 가중치, 지표와 생성 예제가 저장된다.
- 신경 상태는 대화 쌍마다 초기화한다. 서로 다른 대화의 기억은 섞이지 않는다.
- 시간 상수는 모델 가정에서 학습한 값이며 실제 초파리 전기생리 측정값이 아니다.